# Imports

In [ ]:
import subprocess, sys

DEPS = [
    ('pandas',            'pandas'),
    ('numpy',             'numpy'),
    ('requests',          'requests'),
    ('matplotlib',        'matplotlib'),
    ('seaborn',           'seaborn'),
    ('pyarrow',           'pyarrow'),
    ('epiweeks',          'epiweeks'),
    ('scipy',             'scipy'),
    ('statsmodels',       'statsmodels'),
    ('openmeteo-requests','openmeteo_requests'),
    ('requests-cache',    'requests_cache'),
    ('retry-requests',    'retry_requests'),
]

print(' Verificando dependências...')
for pkg, imp_name in DEPS:
    try:
        __import__(imp_name)
        print(f'  ✔ {pkg}')
    except ImportError:
        print(f'  Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'  ✔ {pkg} instalado')

print('\n ✅ Todas as dependências prontas!')

In [ ]:
# 1. Bibliotecas padrão do Python
import gc
import io
import os
import sys
import time
import warnings
from datetime import datetime, date
from pathlib import Path

# 2. Manipulação e análise de dados
import numpy as np
import pandas as pd

# 3. Coleta de dados
import requests
import zipfile
import openmeteo_requests
import requests_cache
from retry_requests import retry

# 4. Visualização de dados
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch as _Patch

# 5. Estatística, datas epidemiológicas e outros utilitários
import datetime as _dt
from epiweeks import Week
from scipy import stats
from scipy.stats import spearmanr

# 6. Statsmodels (usado nas análises complementares)
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
except ImportError:
    import subprocess, sys as _sys
    subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
    from statsmodels.tsa.seasonal import seasonal_decompose

# 7. Google Colab (para montar o drive)
from google.colab import drive

print('✅ Imports configurados')

# Constantes do projeto

In [ ]:
# -- Doenças SINAN --
DOENCAS_SINAN = {
    'DENG': 'Dengue',
    'CHIK': 'Chikungunya',
    'ZIKA': 'Zikavirus',
}

# -- Municípios --
MUNICIPIOS = {
    # Oeste potiguar
    '240940': 'PAU DOS FERROS', # 2409407
    '240800': 'MOSSORO',        # 2408003
    # Leste litorâneo
    '240810': 'NATAL',          # 2408102
    '241440': 'TOUROS',         # 2414407
    # Central potiguar
    '240200': 'CAICO',          # 2402006
    '240720': 'MACAU',          # 2407203
    # Agreste
    '241120': 'SANTA CRUZ',     # 2411205
    '240580': 'JOAO CAMARA'     # 2405801
}

# -- Período de análise --
ANOS        = list(range(2002, 2022))
DATA_INICIO = '2002-01-01'
DATA_FIM    = '2022-12-31'

# -- URLs --
URL_BASE_SINAN = (
    'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br'
    '/SINAN/{doenca_nome}/csv/{doenca_sigla}BR{ano_str}.csv.zip'
)